# 01 — Exploratory data analysis: IBM AML HI-Small

Each section follows **question → code → output → interpretation**. Decisions that come out of a section move into `src/` as functions with tests; this notebook keeps the evidence and the reasoning.

**Public repo rule:** outputs show aggregates and plots only, never raw rows.

## 0. Setup

In [1]:
import pandas as pd

from skeptical_second_reader.data import (
    ID_COLUMNS,
    KEY_SEPARATOR,
    account_key,
    load_transactions,
)

## 1. Load

**Question:** does the loader keep the data intact? Two traps in the raw file: both account columns are named `Account`, and bank codes carry leading zeros (`010`), which the pandas pyarrow engine silently drops.

The first call reads the CSV (slow) and caches it as parquet in `data/interim/`; later calls read the parquet.

In [2]:
%%time
df = load_transactions()
print(f"rows: {len(df):,}  columns: {df.shape[1]}")

rows: 5,078,345  columns: 11
CPU times: total: 3.25 s
Wall time: 496 ms


In [3]:
df.dtypes

timestamp             datetime64[us]
from_bank                        str
from_account                     str
to_bank                          str
to_account                       str
amount_received              float64
receiving_currency               str
amount_paid                  float64
payment_currency                 str
payment_format                   str
is_laundering                  int64
dtype: object

In [4]:
# Invariants of the loaded data; the notebook stops here if one fails.
assert df.isna().sum().sum() == 0, "missing values"
assert set(df["is_laundering"].unique()) <= {0, 1}, "label is not binary"
for col in ID_COLUMNS:
    assert pd.api.types.is_string_dtype(df[col]), f"{col} is not a string column"
assert df["from_bank"].str.startswith("0").any(), "leading zeros were dropped"

print("time span:", df["timestamp"].min(), "->", df["timestamp"].max())
print(f"laundering rate: {df['is_laundering'].mean():.4%}")

time span: 2022-09-01 00:00:00 -> 2022-09-18 16:18:00
laundering rate: 0.1019%


**Interpretation:** The data loaded correctly: all invariant checks passed (no missing values, binary label, ID columns stored as strings with leading zeros intact). The dataset contains 5,078,345 transactions covering about 18 days (1–18 September 2022). The laundering rate is 0.1019%, roughly 1 in 980 transactions, so the classes are highly imbalanced. A model that labels every transaction as clean would reach about 99.9% accuracy while catching no laundering at all. Accuracy is therefore not informative here; we will use AUC-PR (area under the precision–recall curve) as the main metric.

## 2. Account identity

**Question:** what identifies an account? Each transaction names a sender and a receiver, each as a (bank code, account number) pair. Two checks decide the identity used everywhere downstream (features, split, cases):

1. Would numeric bank codes merge different banks? (Why IDs are strings.)
2. Does the same account number occur at more than one bank? (Whether the number alone is enough.)

In [5]:
# Check 1: bank codes that would collide if read as numbers.
banks = pd.Series(pd.concat([df["from_bank"], df["to_bank"]]).unique())
as_number = banks.str.lstrip("0").replace("", "0")
codes_per_number = banks.groupby(as_number).nunique()

print(f"distinct bank codes: {len(banks):,}")
print(f"codes starting with 0: {banks.str.startswith('0').sum():,}")
print(f"numeric values shared by 2+ codes: {(codes_per_number > 1).sum():,}")

distinct bank codes: 30,528
codes starting with 0: 30,528
numeric values shared by 2+ codes: 58


In [6]:
# Check 2: account numbers that occur at more than one bank.
pairs = pd.concat(
    [
        df[["from_bank", "from_account"]].set_axis(["bank", "account"], axis=1),
        df[["to_bank", "to_account"]].set_axis(["bank", "account"], axis=1),
    ]
).drop_duplicates()
banks_per_number = pairs.groupby("account")["bank"].nunique()
shared = banks_per_number[banks_per_number > 1]

print(f"distinct (bank, account) pairs: {len(pairs):,}")
print(f"distinct account numbers:       {banks_per_number.size:,}")
print(f"numbers at 2+ banks:            {shared.size:,}")
print(f"most banks for one number:      {shared.max() if shared.size else 0}")

distinct (bank, account) pairs: 515,088
distinct account numbers:       515,080
numbers at 2+ banks:            8
most banks for one number:      2


In [7]:
# The account key used downstream: (bank, account number) joined by a separator.
for col in ID_COLUMNS:
    has_separator = df[col].str.contains(KEY_SEPARATOR, regex=False).any()
    assert not has_separator, f"separator in {col}"

df["from_key"] = account_key(df["from_bank"], df["from_account"])
df["to_key"] = account_key(df["to_bank"], df["to_account"])
n_keys = pd.concat([df["from_key"], df["to_key"]]).nunique()

assert n_keys == len(pairs), "key count differs from distinct (bank, account) pairs"
print(f"distinct account keys: {n_keys:,}")

distinct account keys: 515,088


**Interpretation:** There are 515,088 unique accounts in this dataset. Only 8 of the 515,080 account numbers occur at more than one bank, and each of them at most at two banks. For this reason we identify an account not just by its account number but by the (bank code, account number) pair; otherwise, for these 8 numbers, two different accounts would be merged into one. We accept that the effect is small, but the identity should follow from what an account is, not from what this dataset happens to contain, and it costs only a separator character. A unit test checks that an account gets the same key as sender and as receiver, so one account cannot appear under two keys, which would let it fall into both the training and the test set.